In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#时间归一化‘缺失值、异常值处理
df = pd.read_csv('UserBehavior.csv',header=None)
df.columns = ['user_id', 'item_id', 'category_id','behavior_type', 'timestamp']
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s',errors='coerce')
df = df[(df['datetime'] >= 'xxxx-xx-xx') & (df['datetime'] <= 'xxxx-xx-xx')]
print(df.head())
print(df.info())
print(df.tail())
print(df.describe())
print(df['datetime'].min())
print(df['datetime'].max())
print(df.isnull().sum())
#允许根据datetime自由返回时间范围的筛选
def filter_by_date(df, start, end):
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)
    filtered_df = df[(df['datetime'] >= start) & (df['datetime'] < end + pd.Timedelta(days=1))]
    return filtered_df.sort_values('datetime')

start_date = input("请输入开始日期 (格式YYYY-MM-DD): ")
end_date = input("请输入结束日期 (格式YYYY-MM-DD): ")

filtered_df = filter_by_date(df, start_date, end_date)

print(f"\n筛选日期范围: {start_date} 到 {end_date} 的数据如下：")
print(filtered_df)

# 统计截取数据集用户数，商品数，商品类别数，用户行为数，总条目数
user_count = filtered_df['user_id'].nunique()
item_count = filtered_df['item_id'].nunique()
category_count = filtered_df['category_id'].nunique()
behavior_count= filtered_df['behavior_type'].value_counts()
behavior_count_total = filtered_df['behavior_type'].value_counts().sum()
total_rows = len(filtered_df)

summary = f"""
数据摘要：
- 用户数: {user_count}
- 商品数: {item_count}
- 商品类别数：: {category_count}
- 用户行为总数: {behavior_count_total}
  - 浏览（pv）: {behavior_count.get('pv', 0)}
  - 购买（buy）: {behavior_count.get('buy', 0)}
  - 加购（cart）: {behavior_count.get('cart', 0)}
  - 收藏（fav）: {behavior_count.get('fav', 0)}
- 总条目数: {total_rows}
"""
print(summary)

# 分别求得浏览、购买数前10的商品，并展示其类别
top10_pv_item = filtered_df[filtered_df['behavior_type'] == 'pv'].groupby('item_id').size().sort_values(ascending=False).head(10)
top10_buy_item = filtered_df[filtered_df['behavior_type'] == 'buy'].groupby('item_id').size().sort_values(ascending=False).head(10)
print(f"前十浏览量商品:\n{top10_pv_item}")
print(f"前十购买量商品:\n{top10_buy_item}")

top10_pv_cate = filtered_df[filtered_df['item_id'].isin(top10_pv_item.index)][['item_id', 'category_id']].drop_duplicates()
top10_buy_cate = filtered_df[filtered_df['item_id'].isin(top10_buy_item.index)][['item_id', 'category_id']].drop_duplicates()

top10_pv_df = top10_pv_item.rename('pv_count').reset_index()
top10_buy_df = top10_buy_item.rename('buy_count').reset_index()

top10_pv = top10_pv_df.merge(top10_pv_cate, on='item_id', how='left').sort_values(by='pv_count', ascending=False).reset_index(drop=True)
top10_buy = top10_buy_df.merge(top10_buy_cate, on='item_id', how='left').sort_values(by='buy_count', ascending=False).reset_index(drop=True)

top10_pv['rank'] = top10_pv.index + 1
top10_buy['rank'] = top10_buy.index + 1

print(f"前十浏览量商品及其类别:\n{top10_pv[['rank', 'item_id', 'category_id', 'pv_count']].to_string(index=False)}")
print(f"前十购买量商品及其类别:\n{top10_buy[['rank', 'item_id', 'category_id', 'buy_count']].to_string(index=False)}")